# 1강: 워드 임베딩과 순환신경망 기반 모델

# 1. 워드 임베딩

# 1-1. 원-핫 인코딩

원-핫 인코딩이란?
- 규칙 기반 혹은 통계적 자연어 처리 연구의 대다수는 단어를 원자적(쪼갤 수 없는) 기호로 취급
  - 예: hotel, conference, walk 같은 단어들
- 벡터 공간 관점에서 보면, 이는 한 원소만 1이고 나머지는 모두 0인 벡터를 의미한다.
  - [00000000000000010000]
- 차원 수(= 단어 사전 크기)는 대략 다음과 같다:
  - 음성 데이터(2만) - Penn Treebank(PTB) 코퍼스(5만) - big vocab(50만) - Google 1T(1300만)
- 이를 원-핫(one-hot) 표현이라고 부르고, 단어를 원-핫 표현으로 바꾸는 과정을 원-핫 인코딩이라고 한다.

원 핫 인코딩의 문제점
- 예: 웹 검색
  - [삼성 노트북 배터리 사이즈] == [삼성 노트북 배터리 용량]
  - [갤럭시 핸드폰] == [갤럭시 스마트폰]
  - 핸드폰 [00000000000000000100000]
  - 스마트폰 [0000100000000000000000] = 0
- 검색 쿼리 벡터와 대상이 되는 문서 벡터들이 서로 직교하게 되어, 원-핫 벡터로는 유사도를 측정할 수 없다.
- 원-핫 인코딩과 같은 전통적인 텍스트 표현 방식에는 여러 한계가 존재한다.
  - 1. 차원의 저주(Curse of Dimensionality)
    - 고차원의 희소 벡터를 다루기 위해선 많은 메모리가 필요하다.
    - 차원이 커질수록 데이터가 점점 더 희소해져서 활용이 어렵다.
  - 2. 의미적 정보 부족
    - 비슷한 단어라도 유사한 벡터로 표현되지 않는다.
    - 예: '은행'과 '금융'은 의미적으로 밀접하지만, 원-핫 인코딩에서는 전혀 무관한 벡터로 취급된다.

# 1-2. 워드 임베딩

주변 단어들을 활용해보기
- 단어를 주변 단어들로 표현하면, 많은 의미르 담을 수 있다.
- 현대 통계적 자연어처리에서의 가장 성공적인 아이디어 중 하나이다.

워드 임베딩이란?
- 단어를 단어들 사이의 의미적 관계를 포착할 수 있는 밀집(dense)되고, 연속적 / 분산적(distributed) 벡터 표현으로 나타내는 방법이다.
  - 원-핫 인코딩에선 은행과 금융이 완전히 독립적인(무관한) 벡터로 표현되었지만, 
  - 워드 임베딩에선 두 단어의 벡터가 공간상 서로 가깝게 위치하며, 이를 통해 의미적 유사성을 반영할 수 있다.

대표적인 워드 임베딩 기법 - Word2Vec
- Word2Vec은 2013년 Google에서 개발한 워드 임베딩 기법
- 단어의 표현을 간단한 인공 신경망을 이용해서 학습

Word2Vec의 아이디어
- Word2Vec의 아이디어는 각 단어와 그 주변 단어들 간의 관계를 예측한다는 것이다.
- Word2Vec엔 두 가지 알고리즘이 존재한다.
  - Skip-grams(SG) 방식
    - 중심 단어를 통해 주변 단어들을 예측하는 방법이다.
    - 단어의 위치(앞 / 뒤)에 크게 구애 받지 않는다.
  - Continuous Bag of Words(CBOW) 방식
    - 주변 단어들을 통해 중심 단어를 예측하는 방법이다.
    - 문맥 단어들의 집합으로 중심 단어를 맞춘다.

Skip-grams(SG): 중심 단어를 통해 주변 단어 예측하기
- 윈도우 크기(window size) = 중심 단어 주변 몇 개 단어를 문맥으로 볼 것인가?
- 예: (윈도우 크기 = 2)
  - 문장: '...problems turning into banking crises as ...'
  - 중심 단어 'banking' (위치 t)
  - 주변 단어 = {'turning', 'into', 'crises', 'as'}

Continuous Bag of Words(CBOW): 주변 단어를 통해 중심 단어 예측하기
- 목표: 주변 단어들의 집합이 주어졌을 때, 그 문맥과 함께 등장할 수 있는 단일 단어를 예측한다.

Skip-Gram
- 장점
  - 적은 데이터에도 잘 동작한다.
  - 희귀 단어나 구 표현에 강하다
- 단점
  - 학습 속도가 느리다

CBOW
- 장점
  - 학습 속도가 빠르다
  - 자주 나오는 단어에 강하다
- 단점
  - 희귀 단어 표현에 약하다

# 2. 순차적 데이터

순차적 데이터란 무엇인가?
- 자연엔 수 많은 순차적 데이터(Sequential Data)가 존재한다.
- 특징
  - 1. 순서가 중요하다.
    - 데이터의 순서가 바뀌면 의미가 달라진다.
    - 예: 나는 너를 사랑해 /= 너는 나를 사랑해
  - 2. 장기 의존성(Long-term dependency)
    - 멀리 떨어진 과거의 정보가 현재 / 미래에 영향을 준다
    - 예: 여러 개의 문 중 파란 문을 열고 안으로 들어가면 너는 (?)를 찾게 될거야
  - 3. 가변 길이(Variable length)
    - 순차 데이터는 길이가 일정하지 않고, 단어 수도 제각각이다.

순차적 데이터를 처리하려면?
- 따라서, 순차적 데이터를 처리하려면 일반적인 모델들(예: 선형회귀, MLP 등)로는 불가능하다.
- Sequential Models이 필요하다.
  - 예: RNN, LSTM, Transformer 등 

# 3. RNN

전통적인 인공신경망
- 전통적인 인공신경망(MLP, CNN)들은 고정된 길이의 압력을 받아 가변 길이의 데이터를 처리하기에 적합하지 않다.

RNN이란?
- 하지만, RNN은 가변 길이의 입력을 받을 수 있고, 이전 입력을 기억할 수 있기 때문에, 순차적 데이터 처리에 적합한 아키텍처이다.

RNN 아키텍처 설명
- 전통적인 신경망(MLP, CNN등)과 달리, RNN은 이전 시점의 정보를 담는 hidden state를 가지고 있다.
- 따라서, 입력 시퀀스 벡터 x를 처리할 때, 각 시점마다 recurrence 수식을 적용하여 hidden state를 업데이트 한다.

RNN의 특징
- RNN은 한 번에 하나의 요소를 처리하고, 정보를 앞으로 전달한다.
- 펼쳐서 보면, RNN은 각 층이 하나의 시점을 나타내는 깊은 신경망처럼 보인다.
- RNN은 hidden state를 유지하면서 가변 길이 데이터를 처리할 수 있다.
- RNN의 출력은 과거 입력에 영향을 받는다는 점에서, feedforward 신경망과 다르다.

RNN의 한계: 기울기 소실(vanishing gradient) 문제
- 기울기 소실 문제란?
  - 딥러닝에서 역전파 시 앞쪽 층의 기울기가 0에 가까워져서 장기 의존성 학습이 어려워 지는 현상
- 왜 일어날까?
 - 1. 역전파 과정에서 작은 값들이 계속 곱해진다.
 - 2. 과거 시점에서 온 오차 신호는 갈수록 더 작은 기울기를 갖는다.
 - 3. 결국 파라미터들이 장기 의존성은 학습하지 못하고, 단기 의존성만 포착하게 된다.

# 4. LSTMs

LSTMs란?
- 기울기 소실 문제를 해결하기 위해, 1997년에 제안된 RNN의 한 종류
- LSTMs의 특징
  - 시점 t에서 RNN은 길이가 n인 벡터 hidden state ht와 cell state ct를 가진다.
    - hidden state는 short-term information을 저장한다.
    - Cell state는 long-term information을 저장한다.
  - LSTMs는 cell state에서 정보를 읽고(read), 지우고(erase), 기록(write)할 수 있다.

LSTMs
- 3가지 게이트를 통해 어떤 정보를 지우고, 쓰고, 읽을지 결정한다.
  - Forget gate: 이전 cell state에서 무엇을 버리고 무엇을 유지할 지 결정
  - Input gate: 새 정보 중 얼마나 cell state에 쓸지 결정
  - Output gate: cell state 중 얼마나 hidden state로 내보낼지 결정
- 게이트의 동작?
  - 매 시점마다 게이트의 각 요소는 열림(1), 닫힘(0), 혹은 그 사이 값으로 설정된다.
  - 게이트는 동적으로 계산되며, 현재 입력과 hidden state 등 문맥에 따라 값이 정해진다.

# 2강: 자연어 생성 모델

# 1. 언어 모델이란?

언어모델이란?
- 언어모델이란 인간의 두뇌가 자연어를 생성하는 능력을 모방한 모델이다
  - 단어 시퀀스 전체에 확률을 부여하여 문장의 자연스러움을 측정한다.
- 한 문장의 확률은 각 단어의 조건부 확률들의 곱으로 표현할 수 있다.


대표적인 언어모델 -N-gram 언어모델
- n-gram이란, 연속된 n개의 단어 묶음을 말한다.
  - unigrams: 'The', 'students', 'opened', 'their'
  - bigrams: 'The students', 'students opened', 'opened their'
  - trigrams: 'The students opened', 'students opened their'
  - four-grams: 'The students opened their'
- 다양한 n-gram이 얼마나 자주 등장하는지 통계를 수집하고, 이를 활용해 다음 단어를 예측한다.

언어모델 사용 예시: Statistical Machine Translation
- 1990년부터 2010년까지는 Machine Translation을 통계학적으로 접근했다.
- 예시: 한국어 -> 영어
  - 한국어 문장 x가 주어졌을 때, 가장 잘 맞는 영어 문장 y를 찾아야 한다.
  - Bayes Rule을 이용해, 식을 두 부분으로 쪼개어 번역모델과 언어모델로 나누는 방법을 사용했다.

# 2. Seq2Seq

Neural Machine Translation

Neural Machine Translation이란?
- Neural Machine Translation이란 인공 신경말을 이용해 기계 번역을 수행하는 방법이다.
- 이 때 사용되는 신경망 구조를 sequence-to-sequence(Seq2Seq)이라 하며, 두 개의 RNNs로 이루어진다.
  - 2014년 Google의 'Sequence to Sequence Learning with Neural Networks'라는 논문에서 처음 소개됨

Seq2Seq Architecture
- Seq2Seq는 Encoder와 Decoder로 이루어진다.
  - Encoder는 입력 문장에 담긴 정보를 인코딩한다.
  - Decoder는 인코딩된 정보를 조건으로 하여 타겟 문장(출력)을 생성한다.

Seq2Seq의 다양한 적용
- Seq2Seq 구조는 기계번역 외에도 다양한 태스크에 적용 가능하다
  - 요약: 긴 길이의 문서를 읽고, 짧은 길이의 문장으로 요약된 텍스트를 출력하는 태스크
  - 대화: 사용자의 발화를 기반으로, 맥락에 맞는 대답(출력 텍스트)을 생성하는 태스크
  - 코드 생성: 자연어로 작성된 설명 혹은 명령어를 입력받아, 그에 대응하는 프로그래밍 코드 혹은 쿼리를 출력하는 태스크

Seq2Seq 학습 수행
- Seq2Seq 모델은 인코더와 디코더가 하나의 통합 네트워크로 연결되어 있다.
- 디코더에서 발생한 오차는 역전파 과정을 통해 입력을 처리한 인코더까지 전달되어 전체 네트워크가 End-to-End로 동시에 최적화 된다

Seq2Seq 학습 수행(Teacher Forcing)
- 학습 초반에는 모델의 예측 능력이 떨어지기 때문에 학습이 불안정 할 수 있다.
- Teacher Forcing이란?
  - 모델이 스스로 예측한 단어 대신 정답 단어를 디코더 입력으로 강제로 넣어줌으로써 훨씬 안정적이고 빠르게 학습을 수행하는 방법이다.

Seq2Seq의 토큰 출력 방법(Greedy Inference)
- 토큰을 출력하는 방법 중 하나로, 각 단계에서 가장 확률이 높은 단어를 선택한다.
- 한계
  - 되돌리기가 불가능하다.

Seq2Seq의 토큰 출력 방법(Beam Search)
- Beam Search
  - 1. 매 단계마다 k개의 가장 유망한 후보 유지
  - 2. 후보가 <EOS>에 도달하면, 완성된 문장으로 리스트 추가
  - 3. <EOS>문장이 충분히 모이면 탐색 종료
  - 4. 각 후보들의 점수를 로그 확률의 합으로 구해 최종 선택

# 3. Attention

Seq2Seq의 한계: the bottleneck problem
- Bottleneck problem이란?
  - 인코더는 입력 문장 전체를 하나의 벡터로 요약하는데, 마지막 hidden state에 문장의 모든 정보가 담긴다.
  - 고정 길이 벡터 하나에 모든 문장의 의미를 압축하다 보니 정보 손실이 생길 수 있는데, 이를 bottleneck problem 이라고 한다.

Attention의 인사이트
- Attention은 디코더가 단어를 생성할 떄, 인코더 전체 hidden state 중 필요한 부분을 직접 참조할 수 있도록 한다.
- 즉, 매 타임스텝마다 '어떤 단어 / 구절에 집중할 지'를 가중치로 계산해, bottleneck 문제를 완화했다.

Attention의 효과
- Attention mechanism은 많은 장점이 존재한다.
  - 1. NMT 성능 향상
    - 디코더가 소스 문장 전체가 아닌, 필요한 부분에만 집중할 수 있기 때문이다.
  - 2. Bottleneck Problem 해결
    - 디코더가 인코더의 모든 hidden state에 직접 접근할 수 있다.
  - 3. Vanishing Gradient Problem 완화
    - Attention은 멀리 떨어진 단어도 직접 연결할 수 있게 해준다.
- BLEU score: 기계 번역의 출력이 사람 번역과 얼마나 비슷한 지 평가하는 지표

Attention의 효과: 해석 가능성(Interpretability)
- Attention 분포를 보면, decoder가 어떤 단어를 생성할 때, 입력 문장의 어느 부분에 집중했는지 확인할 수 있다.
- 즉, 모델이 내부적으로 참고한 근거를 사람이 파악할 수 있다. 
  - 모델의 의사결정 과정을 해석할 수 있는 단서

Attention의 효과: 정렬(Alignment)
- 기계번역에서는 전통적으로 단어 alignment모델을 따로 학습해야 했다.
- 하지만, attention을 통해 decoder가 필요한 입력 단어에 자동으로 집중하기 때문에, 단어와 단어 간의 매핑 관계를 자연스럽게 학습한다.

Attention: Query와 Values
- Seq2Seq에서 attention을 사용할 때, 각 decoder의 hidden state와 모든 encoder의 hidden state 간의 관계를 Query와 Values외 관계로 볼 수 있다.
- 이 관점에서 Attention 과정을 정리해보면:
  - 1. Query와 Values 사이 유사도 점수(score) 계산 (예: dot-product, multiplication, additive 등)
  - 2. Softmax를 통해 확률 분포(attention distribution) 얻기
  - 3. 분포를 이용해 values를 가중합 -> context vector (attention output)